# 挑战大模型工程师 · 阶段一「造」：从零手写 mini-GPT

不用任何现成模型类，只用 PyTorch 张量，写出分词器、Transformer 和训练循环，训练一个 **500 万到 5000 万参数**的中文模型。

**开始前**
1. 在考场网页的阶段一页面 **领取挑战码**：<https://claude.ai/artifact/Gqn9v7mWLPYjJ8SbzNmus7#s1>
2. 菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**。
3. 算力预算（暂定）：Colab T4 训练总耗时 ≤ 120 分钟；Mac ≤ 240 分钟。

**这个模板给你的**：环境检查、语料准备、训练循环的计时和日志、跑裁判。
**要你自己写的**：标了 `TODO` 的部分 —— BPE 分词器、多头因果注意力、Transformer Block、带 KV Cache 的生成。这正是这一关考的内容，模板里没有答案。

用自己的 Mac 做：克隆仓库后用 Jupyter 打开这个文件，Colab 专用的单元格会自动跳过。

In [ ]:
# 1. 环境检查
import torch, platform
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if torch.cuda.is_available():
    DEVICE = "cuda"; print("GPU:", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"; print("Apple Silicon MPS")
else:
    DEVICE = "cpu"; print("⚠️ 没有 GPU。Colab 请把运行时类型改成 T4 GPU；裁判会核对运行设备。")
print("torch", torch.__version__, "| python", platform.python_version())

In [ ]:
# 2. 取任务包，建工作目录（my_gpt.py、train_log.json、grad_judge.py 要放在同一个文件夹）
import os, shutil, pathlib
WORK = pathlib.Path("/content/work" if IN_COLAB else "work").resolve()
WORK.mkdir(parents=True, exist_ok=True)
if IN_COLAB and not pathlib.Path("/content/llm-challenge").exists():
    !git clone --depth 1 https://github.com/kulakeepor/llm-challenge.git /content/llm-challenge
if IN_COLAB:
    PACK = pathlib.Path("/content/llm-challenge")
else:  # 本地：从当前目录往上找仓库根目录
    PACK = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "judge" / "grad_judge.py").exists())
shutil.copy(PACK / "judge" / "grad_judge.py", WORK / "grad_judge.py")
os.chdir(WORK); print("工作目录：", WORK)

## 准备语料

准备几 MB 的中文纯文本（UTF-8），比如公有领域的古诗文、你自己整理的文章。存成 `corpus.txt`。
Colab 里运行下面的单元格会弹出上传按钮；在 Mac 上直接把文件放进 `work/` 文件夹。

In [ ]:
# 3. 上传 / 读取语料
CORPUS = WORK / "corpus.txt"
if IN_COLAB and not CORPUS.exists():
    from google.colab import files
    up = files.upload()
    name = next(iter(up)); (WORK / name).rename(CORPUS)
text = CORPUS.read_text(encoding="utf-8")
print(f"{len(text):,} 个字符，{len(set(text)):,} 种字符")
split = int(len(text) * 0.9)
train_text, val_text = text[:split], text[split:]

## 写 my_gpt.py

裁判只通过这一个文件认识你的模型，接口约定（详见仓库 `docs/my_gpt_interface.md`）：

```python
def load() -> (model, meta)
    # model(idx)                                  -> logits [B, T, vocab_size]
    # model.generate(idx, max_new_tokens, use_cache) -> 贪心生成 [B, T + max_new_tokens]
    # meta = {"vocab_size": int, "block_size": int}
```

裁判会检查：参数量在 500 万到 5000 万之间；形状正确；**因果性**（改动第 t 个字之后的内容，前 t 个位置的输出不能变）；**带和不带 KV Cache 的贪心生成必须完全一致**。

下面的单元格把代码写进 `my_gpt.py`。规则：不准 `import transformers` 或任何现成的注意力 / Transformer 模块（如 `nn.MultiheadAttention`、`nn.TransformerEncoder`、`F.scaled_dot_product_attention`），只用基础层（`nn.Linear`、`nn.Embedding`、`nn.LayerNorm` 等）和张量运算。

In [ ]:
%%writefile my_gpt.py
"""阶段一：从零手写 mini-GPT。只用 PyTorch 张量和基础层。"""
import json, math, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = pathlib.Path(__file__).resolve().parent

# ---------------- 分词器 ----------------
class BPETokenizer:
    """字节对编码（BPE）分词器。

    train(text, vocab_size): 从单个字符出发，反复合并出现次数最多的相邻对，直到词表达到 vocab_size。
    encode(text) -> list[int]；decode(ids) -> str；save(path) / load(path)。

    注意：
    - 几 MB 语料上，每次合并都把全文重扫一遍会非常慢。想想怎么只更新受影响的部分。
    - 验证集里可能有训练集没见过的字，encode 时要有处理办法（不能直接报错）。
    """
    def __init__(self):
        self.vocab = {}    # id -> 字符串
        self.merges = []   # 按顺序记录的合并规则

    def train(self, text, vocab_size):
        raise NotImplementedError("TODO：实现 BPE 训练")

    def encode(self, text):
        raise NotImplementedError("TODO：按 merges 的顺序编码")

    def decode(self, ids):
        return "".join(self.vocab[i] for i in ids)

    def save(self, path):
        pathlib.Path(path).write_text(json.dumps({"vocab": self.vocab, "merges": self.merges}, ensure_ascii=False))

    @classmethod
    def load(cls, path):
        d = json.loads(pathlib.Path(path).read_text())
        tok = cls(); tok.vocab = {int(k): v for k, v in d["vocab"].items()}; tok.merges = [tuple(m) for m in d["merges"]]
        return tok

# ---------------- 模型 ----------------
class CausalSelfAttention(nn.Module):
    """多头因果自注意力。forward(x, cache=None) -> (y, new_cache)。
    cache 用来保存之前位置的 K、V，生成时每步只算新位置。"""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        # TODO：定义 Q/K/V 投影、输出投影、因果遮罩
        raise NotImplementedError("TODO：多头因果注意力")

    def forward(self, x, cache=None):
        raise NotImplementedError

class Block(nn.Module):
    """一个 Transformer Block：LayerNorm、注意力、残差、MLP。"""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        raise NotImplementedError("TODO：拼装 Block")

    def forward(self, x, cache=None):
        raise NotImplementedError

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.1):
        super().__init__()
        self.vocab_size, self.block_size = vocab_size, block_size
        self.config = dict(vocab_size=vocab_size, block_size=block_size, n_layer=n_layer, n_head=n_head, n_embd=n_embd, dropout=dropout)
        # TODO：词嵌入、位置编码、n_layer 个 Block、最后的 LayerNorm 和输出层
        raise NotImplementedError("TODO：组装 GPT")

    def forward(self, idx, cache=None, start_pos=0):
        """idx: [B, T] -> logits [B, T, vocab_size]。
        训练和裁判直接调用 model(idx)；生成时可以传 cache 和 start_pos。"""
        raise NotImplementedError

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, use_cache=True):
        """贪心生成（每步取概率最大的字），返回 [B, T + max_new_tokens]。
        use_cache=False：每步把整段序列重新算一遍。
        use_cache=True ：每步只算新位置，复用之前的 K、V。
        两种方式的结果必须一模一样。注意序列超过 block_size 时的处理。"""
        raise NotImplementedError("TODO：带 / 不带 KV Cache 的生成")

# ---------------- 裁判入口 ----------------
def load():
    """裁判调用：返回 (model, meta)。读取训练好的 ckpt.pt。"""
    ck = torch.load(HERE / "ckpt.pt", map_location="cpu")
    model = GPT(**ck["config"])
    model.load_state_dict(ck["model"])
    return model, {"vocab_size": ck["config"]["vocab_size"], "block_size": ck["config"]["block_size"]}

## 训练

下面的循环已经写好计时、验证、保存和日志，你可以改超参数，也可以改优化器和学习率调度。

**每字比特数（bpc）**：验证集上的总负对数似然（自然对数）÷ 验证集的**字符数** ÷ ln 2。
按字符而不是按 token 计算，换了分词器也能公平比较。暂定要求 bpc ≤ 7.0。

参考数据（2026-11-27 在 Colab T4 上实测，语料《全唐诗》370 万字，默认超参数，1940 万参数）：3000 步用时 17.8 分钟，bpc 7.26。
默认的 3000 步**不够**达到 7.0，预算 120 分钟足够你训练更久、调学习率或改模型。分词器训练不计入耗时。

`train_minutes` 由 `time.time()` 实测，写进 `train_log.json`，裁判会读取。改代码时不要手写这个数。

In [ ]:
# 4. 训练
import importlib, math, time, json, torch
import torch.nn.functional as F
import my_gpt; importlib.reload(my_gpt)

VOCAB_SIZE = len(set(train_text)) + 2000   # 词表 = 语料里的字种数 + 合并次数（中文字种常有几千个，必须大于它）
BLOCK_SIZE = 128       # 上下文长度
BATCH_SIZE = 64
MAX_STEPS  = 3000
EVAL_EVERY = 250
LR         = 3e-4

tok = my_gpt.BPETokenizer(); tok.train(train_text, VOCAB_SIZE); tok.save("tokenizer.json")
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids   = torch.tensor(tok.encode(val_text), dtype=torch.long)
print(f"训练 {len(train_ids):,} tokens，验证 {len(val_ids):,} tokens")

model = my_gpt.GPT(len(tok.vocab), BLOCK_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"参数量 {n_params/1e6:.1f}M（要求 5M–50M）")
opt = torch.optim.AdamW(model.parameters(), lr=LR)

def batch(ids):
    ix = torch.randint(len(ids) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([ids[i:i + BLOCK_SIZE] for i in ix]); y = torch.stack([ids[i + 1:i + 1 + BLOCK_SIZE] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def val_bpc():
    model.eval(); nll = 0.0
    for i in range(0, len(val_ids) - 1, BLOCK_SIZE):
        x = val_ids[i:i + BLOCK_SIZE].unsqueeze(0).to(DEVICE); y = val_ids[i + 1:i + 1 + BLOCK_SIZE].unsqueeze(0).to(DEVICE)
        x = x[:, :y.shape[1]]
        logits = model(x); logits = logits[0] if isinstance(logits, (tuple, list)) else logits
        nll += F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), reduction="sum").item()
    model.train()
    return nll / len(val_text) / math.log(2)

history, t0 = [], time.time()
for step in range(1, MAX_STEPS + 1):
    x, y = batch(train_ids)
    logits = model(x); logits = logits[0] if isinstance(logits, (tuple, list)) else logits
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        bpc = val_bpc(); minutes = (time.time() - t0) / 60
        history.append({"step": step, "train_loss": round(loss.item(), 4), "val_bpc": round(bpc, 4), "minutes": round(minutes, 2)})
        print(history[-1])
        torch.save({"config": model.config, "model": model.state_dict()}, "ckpt.pt")
        json.dump({"val_bpc": bpc, "train_minutes": minutes, "history": history}, open("train_log.json", "w"), ensure_ascii=False, indent=1)
print("训练结束，总耗时", round((time.time() - t0) / 60, 1), "分钟")

In [ ]:
# 5. 看看它写了什么（交付说明里要贴 3 段生成样例）
model.eval()
start = torch.tensor([tok.encode(val_text[:8])], dtype=torch.long, device=DEVICE)
print(tok.decode(model.generate(start, 80, use_cache=True)[0].tolist()))

## 跑裁判，生成成绩单

把网页上领到的挑战码填进去。**每次正式提交都用新挑战码**：裁判的测试输入由挑战码决定，没法提前准备。

In [ ]:
# 6. 本地裁判
CHALLENGE = "在网页上领取的挑战码"   # ← 改成你的 8 位挑战码
!python grad_judge.py s1 --challenge $CHALLENGE

In [ ]:
# 7. 下载成绩单，粘贴或上传到网页「提交」页
if IN_COLAB:
    from google.colab import files
    files.download("report_s1.json")

## 交三样证据

1. **成绩单**：`report_s1.json`，在网页阶段一的「提交」页粘贴或上传。
2. **公开仓库**：把 `my_gpt.py`、训练代码、`train_log.json`、loss 曲线和 README（环境、数据来源、一键复现命令）推到你自己的公开 GitHub 仓库，把地址填进网页。
3. **交付说明**：3 段生成样例；训练 / 验证 loss 曲线的走势（什么时候开始过拟合、你怎么处理的）；模型配置。AI 考官批改，70 分及格。

三样都过，盖章通关。